In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

## packages

"""
Created on January 29 08:24:51 2025
@author: alcantar
modifed by J. Hambalek on 18 July 2025
example run 1: python 09_neutral_drift_analysis.py -r ../../minibinders_orthorep_data/ngs_raw/jh_001/references/pjh3_nbonly.fasta -c 50
"""
# activate virtual enviroment before running script
# source activate minibinders

import pandas as pd
import tqdm
import math
from matplotlib import pyplot as plt
import scipy.stats as stats
from matplotlib.lines import Line2D
import numpy as np
from sklearn.cluster import KMeans
from matplotlib.ticker import MaxNLocator
from itertools import combinations
import seaborn as sns

from Bio import SeqIO
import argparse

from utils_nd import *

In [2]:
# set up directories to which to point

experiment_id = 'jh_008' # user sets this
min_contexts = 40 # sets desired minimum contexts to be included in graphing

if experiment_id == 'jh_005':
    fasta_name = 'pjh3_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/*'
    score_tag_name = 'lib'
elif experiment_id == 'jh_004':
    fasta_name = 'pmaa23_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1_x'
elif experiment_id == 'jh_006':
    fasta_name = 'pjh3_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1_x'
elif experiment_id == 'jh_007':
    fasta_name = 'pmaa23_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1'
    score_tag_name2 = 'rep2'
elif experiment_id == 'jh_008':
    fasta_name = 'pmaa23_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1'
    score_tag_name2 = 'rep2'
    score_tag_name3 = 'rep3'
elif experiment_id == 'jh_009':
    fasta_name = 'pjh3_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1'
    score_tag_name2 = 'rep2'
elif experiment_id == 'jh_011':
    fasta_name = 'pjh1_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1'
    score_tag_name2 = 'rep2'
elif experiment_id == 'jh_012':
    fasta_name = 'pjh2_nbonly'
    mutation_df_paths = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{experiment_id}/neutral_drift_library/'
    score_tag_name = 'rep1'
    score_tag_name2 = 'rep2'

# csv_files = glob.glob(os.path.join(mutation_df_paths, f"*scores*.csv"))
# print(csv_files)

In [3]:
# parser = argparse.ArgumentParser()
# parser.add_argument('-r', help='Path to reference fasta')
# parser.add_argument('-c', help='counts per million cutoff used for generation of input dataframes')

# args = parser.parse_args()

# ref_fasta_path = args.r #'../../minibinders_data/ngs_raw/jh_001/references/pjh3_nbonly.fasta' # input
ref_fasta_path = f'../../minibinders_orthorep_data/ngs_raw/{experiment_id}/references/{fasta_name}.fasta'
expt_id = ref_fasta_path.split('/')[-3] #experiment id is the entry past the 4th slash
print(f'experiment id: {expt_id}')

# cpm = args.c
cpm = 10

# define parent / wt sequence
ref_fasta = SeqIO.read(ref_fasta_path, "fasta")
parent_dna_seq = ref_fasta.seq
parent_dna_id = ref_fasta.id
# print(f'parental sequence: {parent_dna_seq}')
# define parent / wt sequence



experiment id: jh_008


In [4]:
# read in relevant dataframes (manually typed in here)
# read in relevant dataframes (manually typed in here)
med_rep1_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/15mdbindcpm{cpm}_mutation_analysis.csv'
med_rep2_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/20mdbindcpm{cpm}_mutation_analysis.csv'
med_rep3_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/21mdbindcpm{cpm}_mutation_analysis.csv'

low_rep1_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/15lobindcpm{cpm}_mutation_analysis.csv'
low_rep2_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/20lobindcpm{cpm}_mutation_analysis.csv'
low_rep3_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/21lobindcpm{cpm}_mutation_analysis.csv'

hi_rep1_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/15hibindcpm{cpm}_mutation_analysis.csv'
hi_rep2_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/20hibindcpm{cpm}_mutation_analysis.csv'
hi_rep3_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/21hibindcpm{cpm}_mutation_analysis.csv'

no_rep1_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/15nobindcpm{cpm}_mutation_analysis.csv'
no_rep2_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/20nobindcpm{cpm}_mutation_analysis.csv'
no_rep3_path = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/mutation_dfs/21nobindcpm{cpm}_mutation_analysis.csv'
# display_rep1_path = '../../minibinders_orthorep_data/minibinders_orthorep_outputs/maa_003/mutation_dfs/mb376-drift-displaypos-rep1_mutation_analysis.csv'
# display_rep2_path = '../../minibinders_orthorep_data/minibinders_orthorep_outputs/maa_003/mutation_dfs/mb376-drift-displaypos-rep2_mutation_analysis.csv'
# print([med_rep1_path,low_rep1_path,hi_rep1_path,no_rep1_path])

In [5]:
out_dir = f'../../minibinders_orthorep_data/minibinders_orthorep_outputs/{expt_id}/neutral_drift_library/'
make_dir(out_dir)
out_dir_figs = out_dir+'plots/'
make_dir(out_dir_figs)

med_rep1_df = pd.read_csv(med_rep1_path, index_col=0)
med_rep2_df = pd.read_csv(med_rep2_path, index_col=0)
med_rep3_df = pd.read_csv(med_rep3_path, index_col=0)

low_rep1_df = pd.read_csv(low_rep1_path, index_col=0)
low_rep2_df = pd.read_csv(low_rep2_path, index_col=0)
low_rep3_df = pd.read_csv(low_rep3_path, index_col=0)

hi_rep1_df = pd.read_csv(hi_rep1_path, index_col=0)
hi_rep2_df = pd.read_csv(hi_rep2_path, index_col=0)
hi_rep3_df = pd.read_csv(hi_rep3_path, index_col=0)

no_rep1_df = pd.read_csv(no_rep1_path, index_col=0)
no_rep2_df = pd.read_csv(no_rep2_path, index_col=0)
no_rep3_df = pd.read_csv(no_rep3_path, index_col=0)


# display_rep1_df = pd.read_csv(display_rep1_path, index_col=0)
# display_rep2_df = pd.read_csv(display_rep2_path, index_col=0)

# print(len(med_rep1_df))
# print(len(low_rep1_df))
# print(len(hi_rep1_df))
# print(len(no_rep1_df))
# print(f'total entries: {np.sum([len(med_rep1_df),len(low_rep1_df),len(hi_rep1_df),len(no_rep1_df)])}')

In [6]:
# consilidate by amino acid sequence
med_rep1_df_agg = aggregate_by_aa_sequence(med_rep1_df)
med_rep2_df_agg = aggregate_by_aa_sequence(med_rep2_df)
med_rep3_df_agg = aggregate_by_aa_sequence(med_rep3_df)

low_rep1_df_agg = aggregate_by_aa_sequence(low_rep1_df)
low_rep2_df_agg = aggregate_by_aa_sequence(low_rep2_df)
low_rep3_df_agg = aggregate_by_aa_sequence(low_rep3_df)

hi_rep1_df_agg = aggregate_by_aa_sequence(hi_rep1_df)
hi_rep2_df_agg = aggregate_by_aa_sequence(hi_rep2_df)
hi_rep3_df_agg = aggregate_by_aa_sequence(hi_rep3_df)

no_rep1_df_agg = aggregate_by_aa_sequence(no_rep1_df)
no_rep2_df_agg = aggregate_by_aa_sequence(no_rep2_df)
no_rep3_df_agg = aggregate_by_aa_sequence(no_rep3_df)

# # display_rep1_df = aggregate_by_aa_sequence(display_rep1_df)
# # display_rep2_df = aggregate_by_aa_sequence(display_rep2_df)

# print(med_rep1_df_agg[med_rep1_df_agg['number_dna_mutations'].apply(lambda x: len(x) >= 10)])
# print(low_rep1_df_agg[low_rep1_df_agg['number_dna_mutations'].apply(lambda x: len(x) >= 10)])
# print(hi_rep1_df_agg[hi_rep1_df_agg['number_dna_mutations'].apply(lambda x: len(x) >= 10)])
# print(no_rep1_df_agg[no_rep1_df_agg['number_dna_mutations'].apply(lambda x: len(x) >= 10)])

# print('entry values:')

# print(len(med_rep1_df_agg))
# print(len(low_rep1_df_agg))
# print(len(hi_rep1_df_agg))
# print(len(no_rep1_df_agg))
# print(f'total entries: {np.sum([len(med_rep1_df_agg),len(low_rep1_df_agg),len(hi_rep1_df_agg),len(no_rep1_df_agg)])}')

# # maybe not ... but worth saving as a separate variable name

In [7]:
# create master dataframe with all sequences that will be considered
# these sequences can be filtered using a read count threshold
# master_rep1_df = pd.concat([med_rep1_df_agg,
#                            low_rep1_df_agg,
#                            hi_rep1_df_agg,
#                            no_rep1_df_agg]).reset_index(drop=True).drop(['read_count'], axis=1)

# master_rep2_df = pd.concat([med_rep2_df_agg,
#                            low_rep2_df_agg,
#                            hi_rep2_df_agg,
#                            no_rep2_df_agg]).reset_index(drop=True).drop(['read_count'], axis=1)

# master_rep3_df = pd.concat([med_rep3_df_agg,
#                            low_rep3_df_agg,
#                            hi_rep3_df_agg,
#                            no_rep3_df_agg]).reset_index(drop=True).drop(['read_count'], axis=1)


master_rep1_df = pd.concat([med_rep1_df_agg,
                           low_rep1_df_agg,
                           hi_rep1_df_agg,
                           no_rep1_df_agg]).reset_index(drop=True)

master_rep2_df = pd.concat([med_rep2_df_agg,
                           low_rep2_df_agg,
                           hi_rep2_df_agg,
                           no_rep2_df_agg]).reset_index(drop=True)

master_rep3_df = pd.concat([med_rep3_df_agg,
                           low_rep3_df_agg,
                           hi_rep3_df_agg,
                           no_rep3_df_agg]).reset_index(drop=True)

# print(master_rep1_df[:5])
print(len(master_rep1_df))
print(len(master_rep2_df))
print(len(master_rep3_df))

37361
119043
100410


In [8]:
# drop duplicate rows with the same amino acid sequence
master_rep1_df_unique = master_rep1_df.drop_duplicates(subset=['aa_sequence'],).reset_index(drop=True)
master_rep2_df_unique = master_rep2_df.drop_duplicates(subset=['aa_sequence'],).reset_index(drop=True)
master_rep3_df_unique = master_rep3_df.drop_duplicates(subset=['aa_sequence'],).reset_index(drop=True)
# print(master_rep1_df_unique[:5])
# print(len(master_rep1_df_unique))

In [9]:
# populate master_dfs with the read counts and also apply read count threshold filter
count_threshold = 2

master_rep1_df_init = master_rep1_df_unique.assign(bin_1=0, bin_2=0, bin_3=0, bin_4=0).drop(['read_count'], axis=1)
master_rep2_df_init = master_rep2_df_unique.assign(bin_1=0, bin_2=0, bin_3=0, bin_4=0).drop(['read_count'], axis=1)
master_rep3_df_init = master_rep3_df_unique.assign(bin_1=0, bin_2=0, bin_3=0, bin_4=0).drop(['read_count'], axis=1)

# print(master_rep1_df_init[:5])
# print(len(master_rep1_df_init))

no_rep1_counts_dict = dict(zip(no_rep1_df_agg['aa_sequence'],no_rep1_df_agg['read_count']))
no_rep2_counts_dict = dict(zip(no_rep2_df_agg['aa_sequence'],no_rep2_df_agg['read_count']))
no_rep3_counts_dict = dict(zip(no_rep3_df_agg['aa_sequence'],no_rep3_df_agg['read_count']))

low_rep1_counts_dict = dict(zip(low_rep1_df_agg['aa_sequence'],low_rep1_df_agg['read_count']))
low_rep2_counts_dict = dict(zip(low_rep2_df_agg['aa_sequence'],low_rep2_df_agg['read_count']))
low_rep3_counts_dict = dict(zip(low_rep3_df_agg['aa_sequence'],low_rep3_df_agg['read_count']))

med_rep1_counts_dict = dict(zip(med_rep1_df_agg['aa_sequence'],med_rep1_df_agg['read_count']))
med_rep2_counts_dict = dict(zip(med_rep2_df_agg['aa_sequence'],med_rep2_df_agg['read_count']))
med_rep3_counts_dict = dict(zip(med_rep3_df_agg['aa_sequence'],med_rep3_df_agg['read_count']))

hi_rep1_counts_dict = dict(zip(hi_rep1_df_agg['aa_sequence'],hi_rep1_df_agg['read_count']))
hi_rep2_counts_dict = dict(zip(hi_rep2_df_agg['aa_sequence'],hi_rep2_df_agg['read_count']))
hi_rep3_counts_dict = dict(zip(hi_rep3_df_agg['aa_sequence'],hi_rep3_df_agg['read_count']))


# print(list(med_rep2_counts_dict.items())[:3])
# print(len(list(no_rep2_counts_dict.keys())))
# print(len(no_rep2_df_agg))
# print(len(list(low_rep2_counts_dict.keys())))
# print(len(low_rep2_df_agg))
# print(len(list(med_rep2_counts_dict.keys())))
# print(len(med_rep2_df_agg))
# print(len(list(hi_rep2_counts_dict.keys())))
# print(len(hi_rep2_df_agg))

In [10]:
raw_counts_rep1_df = retrieve_filter_counts(master_rep1_df_init,
                           no_rep1_counts_dict,
                           low_rep1_counts_dict,
                           med_rep1_counts_dict,
                           hi_rep1_counts_dict,
                           count_threshold=count_threshold)

raw_counts_rep2_df = retrieve_filter_counts(master_rep2_df_init,
                           no_rep2_counts_dict,
                           low_rep2_counts_dict,
                           med_rep2_counts_dict,
                           hi_rep2_counts_dict,
                           count_threshold=count_threshold)
raw_counts_rep3_df = retrieve_filter_counts(master_rep3_df_init,
                           no_rep3_counts_dict,
                           low_rep3_counts_dict,
                           med_rep3_counts_dict,
                           hi_rep3_counts_dict,
                           count_threshold=count_threshold)

In [11]:
test_entry = 801

print(len(raw_counts_rep1_df))
print(raw_counts_rep1_df[test_entry-3:test_entry+3])

test_aa_seq = raw_counts_rep1_df['aa_sequence'][test_entry]
# print(test_aa_seq)

# test_no_count = (test_aa_seq, no_rep1_counts_dict[test_aa_seq]) if test_aa_seq in no_rep1_counts_dict else None
# test_lo_count = (test_aa_seq, low_rep1_counts_dict[test_aa_seq]) if test_aa_seq in low_rep1_counts_dict else None
# test_med_count = (test_aa_seq, med_rep1_counts_dict[test_aa_seq]) if test_aa_seq in med_rep1_counts_dict else None
# test_hi_count = (test_aa_seq, hi_rep1_counts_dict[test_aa_seq]) if test_aa_seq in hi_rep1_counts_dict else None

# print(test_no_count, test_lo_count, test_med_count, test_hi_count)

33225
                                           aa_sequence  \
798  QVQLVESDGRLVQPGGSLRLSCAASEGNVSMLSLGWFRQAPGQGLE...   
799  QVQLVESDGRLVQPGGSLRLSCAASEGNVSMLSLGWFRQTPGQGLE...   
800  QVQLVESDGRLVQPGGSLRLSCAASGGNISMLSLGWFRQAPGQGLE...   
801  QVQLVESDGRLVQPGGSLRLSCAASGGNISMLSLGWFRQAPGQGLE...   
802  QVQLVESDGRLVQPGGSLRLSCAASGGNISMLSLGWFRQAPGQGLE...   
803  QVQLVESDGRLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                          dna_sequence  \
798  [CAGGTCCAACTGGTCGAGAGCGATGGGAGACTGGTGCAGCCAGGC...   
799  [CAGGTCCAACTGGTCGAGAGCGATGGGAGACTGGTGCAGCCAGGC...   
800  [CAGGTCCAACTGGTCGAAAGCGATGGGAGATTGGTGCAACCAGGA...   
801  [CAGGTCCAACTGGTCGAAAGCGATGGGAGATTGGTGCAACCAGGA...   
802  [CAGGTCCAACTGGTCGAAAGCGATGGGAGATTGGTGCAACCAGGA...   
803  [CAGGTCCAACTGGTCGAGAGCGATGGGAGACTGGTGCAGCCAGGC...   

                                         dna_mutations  \
798  [['G23A', 'G28A', 'G77A', 'C149T', 'C177T', 'T...   
799  [['G23A', 'G28A', 'C66T', 'G77A', 'G118A', 'C1...   
800  [

In [12]:
# any sequences lost to read count anomalies get saved here: can be printed to follow lost sequences
raw_counts_rep1_df['sum_reads'] = raw_counts_rep1_df[['bin_1','bin_2','bin_3','bin_4']].sum(axis=1)
raw_counts_rep2_df['sum_reads'] = raw_counts_rep2_df[['bin_1','bin_2','bin_3','bin_4']].sum(axis=1)
raw_counts_rep3_df['sum_reads'] = raw_counts_rep3_df[['bin_1','bin_2','bin_3','bin_4']].sum(axis=1)

raw_counts_ghosts1 = raw_counts_rep1_df[raw_counts_rep1_df['sum_reads'].apply(lambda x: x <= 1)]
raw_counts_ghosts2 = raw_counts_rep2_df[raw_counts_rep2_df['sum_reads'].apply(lambda x: x <= 1)]
raw_counts_ghosts3 = raw_counts_rep3_df[raw_counts_rep3_df['sum_reads'].apply(lambda x: x <= 1)]

# print(raw_counts_ghosts)



In [13]:
# for seq in raw_counts_ghosts['aa_sequence'][:2]:
#     print(seq)
    
#     print('no binding:')
#     print(no_rep1_df['read_count'][no_rep1_df['aa_sequence'] == seq])
#     print(no_rep1_df_agg[no_rep1_df_agg['aa_sequence'] == seq])
#     print(no_rep1_counts_dict[seq] if seq in no_rep1_counts_dict else None)

#     print('------next bin------')

#     print('low binding:')
#     print(low_rep1_df['read_count'][low_rep1_df['aa_sequence'] == seq])
#     print(low_rep1_df_agg[low_rep1_df_agg['aa_sequence'] == seq])
#     print(low_rep1_counts_dict[seq] if seq in low_rep1_counts_dict else None)

#     print('------next bin------')

#     print('med binding:')
#     print(med_rep1_df['read_count'][med_rep1_df['aa_sequence'] == seq])
#     print(med_rep1_df_agg[med_rep1_df_agg['aa_sequence'] == seq])
#     print(med_rep1_counts_dict[seq] if seq in med_rep1_counts_dict else None)

#     print('------next bin------')
    
#     print('high binding:')
#     print(hi_rep1_df['read_count'][hi_rep1_df['aa_sequence'] == seq])
#     print(hi_rep1_df_agg[hi_rep1_df_agg['aa_sequence'] == seq])
#     print(hi_rep1_counts_dict[seq] if seq in hi_rep1_counts_dict else None)

#     print('NEW SEQUENCE:')

In [14]:
# calculate cell fractions in each bin
rep1_cell_counts = [121000, 200000, 133000, 15000]
rep1_cell_fracs = [count/sum(rep1_cell_counts) for count in rep1_cell_counts]

rep2_cell_counts = [118000, 194000, 75000, 12000]
rep2_cell_fracs = [count/sum(rep2_cell_counts) for count in rep2_cell_counts]

rep3_cell_counts = [99935, 150435, 82929, 5597]
rep3_cell_fracs = [count/sum(rep3_cell_counts) for count in rep3_cell_counts]

print(f'rep1 cell fractions: {rep1_cell_fracs}')
print(f'rep2 cell fractions: {rep2_cell_fracs}')
print(f'rep3 cell fractions: {rep3_cell_fracs}')

rep1 cell fractions: [0.2579957356076759, 0.42643923240938164, 0.2835820895522388, 0.031982942430703626]
rep2 cell fractions: [0.2957393483709273, 0.48621553884711777, 0.18796992481203006, 0.03007518796992481]
rep3 cell fractions: [0.2948839762050895, 0.4438972428119541, 0.24470338983050846, 0.016515391152447947]


In [31]:
cpseq_rep1_df = raw_counts_rep1_df.copy()
cpseq_rep2_df = raw_counts_rep2_df.copy()
cpseq_rep3_df = raw_counts_rep3_df.copy()

# sumbins_rep1 = np.sum([cpseq_rep1_df['sum_reads']])
# print(sumbins_rep1)

column_sums_rep1 = raw_counts_rep1_df[['bin_1','bin_2','bin_3','bin_4']].sum().tolist()
print(column_sums_rep1)

cpseq_rep1_df['bin_1seq']= cpseq_rep1_df['bin_1']/column_sums_rep1[0]*rep1_cell_fracs[0]
# print(cpseq_rep1_df['bin_1seq'])

cpseq_rep1_df['bin_2seq']= cpseq_rep1_df['bin_2']/column_sums_rep1[1]*rep1_cell_fracs[1]
# print(cpseq_rep1_df['bin_2seq'])

cpseq_rep1_df['bin_3seq']= cpseq_rep1_df['bin_3']/column_sums_rep1[2]*rep1_cell_fracs[2]
# print(cpseq_rep1_df['bin_3seq'])

cpseq_rep1_df['bin_4seq']= cpseq_rep1_df['bin_4']/column_sums_rep1[3]*rep1_cell_fracs[3]
# print(cpseq_rep1_df['bin_4seq'])

print(cpseq_rep1_df)

print(np.mean([cpseq_rep1_df['bin_1seq']]))
print(np.mean([cpseq_rep1_df['bin_2seq']]))
print(np.mean([cpseq_rep1_df['bin_3seq']]))
print(np.mean([cpseq_rep1_df['bin_4seq']]))


# cpseq_underrep1 = cpseq_rep1_df[cpseq_rep1_df['bin_1seq'].apply(lambda x: x <= 1 and x > 0)]
# print(cpseq_underrep1)


[285682, 277799, 200775, 228556]
                                             aa_sequence  \
0      HVQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1      HVQLVESGGRLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2      QAQLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3      QVKLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4      QVQLFESGGELVQPGGSLRLSCAASEGNVSMLSLGWFRQAPGQGLE...   
...                                                  ...   
33220  QVQLVGSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQVPGQGLE...   
33221  QVQLVGSGGRLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33222  QVQLVKSVGGLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33223  QVQLV_SGGGLIQPGGSLRLSCAASGGNVSMLSLGWYRQAPGQGLE...   
33224  QVQMVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                            dna_sequence  \
0      [CATGTCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGA...   
1      [CATGTCCAACTGGTCGAGAGCGGTGGGAGACTGGTGCAGCCAGGC...   
2      [CAGGCCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
3     

In [17]:
sumbins_rep2 = np.sum([cpseq_rep2_df['sum_reads']])
print(sumbins_rep2)

cpseq_rep2_df['bin_1seq']= cpseq_rep2_df['bin_1']/sumbins_rep2*rep2_cell_counts[0]
# print(cpseq_rep1_df['bin_1seq'])

cpseq_rep2_df['bin_2seq']= cpseq_rep2_df['bin_2']/sumbins_rep2*rep2_cell_counts[1]
# print(cpseq_rep1_df['bin_2seq'])

cpseq_rep2_df['bin_3seq']= cpseq_rep2_df['bin_3']/sumbins_rep2*rep2_cell_counts[2]
# print(cpseq_rep1_df['bin_3seq'])

cpseq_rep2_df['bin_4seq']= cpseq_rep2_df['bin_4']/sumbins_rep2*rep2_cell_counts[3]
# print(cpseq_rep1_df['bin_4seq'])

# print(cpseq_rep1_df)

print(np.mean([cpseq_rep2_df['bin_1seq']]))
print(np.mean([cpseq_rep2_df['bin_2seq']]))
print(np.mean([cpseq_rep2_df['bin_3seq']]))
print(np.mean([cpseq_rep2_df['bin_4seq']]))


cpseq_underrep2 = cpseq_rep2_df[cpseq_rep2_df['bin_4seq'].apply(lambda x: x <= 1 and x > 0)]
print(cpseq_underrep2)


201652
1.0335569310052168
1.351683484146584
0.5810790710929885
0.08050912160612352
                                             aa_sequence  \
25     QVQLVASGGGLVQPGDSLRLSCTASGGNVSMLSLGWFRQAPGQGLE...   
33     QVQLVDSGGGLIQPGGSLRLSCAASGGNISMLSLGWFRQAPGQGLE...   
59     QVQLVEGGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
91     QVQLVENDGGLVQPGGSLRLSCAASEVNVSMLSLGWFRQAPGQGLE...   
101    QVQLVENGGGLIQPDGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
...                                                  ...   
22830  QVQLVGSGGGLVQPGGSLRLSCAASGENVSMLSLGWFRQVQGQGLE...   
22831  QVQLVVSGGGLVQPGDSLRLSCAASEGNVSMLSLDWFRQAPGQGLE...   
22832  QVQLV_SGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
22833  QVRLVESGGGLVQSGGSLRLNCTASGGNVSMLSLGWFRQALGQGLE...   
22834  QV_LVESGGGLVQPGGSLRLSCDASGENVSMLSLGWFRQAPGQGLE...   

                                            dna_sequence  \
25     [CAGGTCCAACTGGTCGCGAGCGGTGGGGGACTGGTACAGCCAGGC...   
33     [CAGGTCCAACTGGTCGATAGCGGAGGGGGGCTGATACAACCAGGC...   
59     [CAGGTCCA

In [18]:
# create new dataframes that will contain normalized scores
norm_counts_rep1_df = raw_counts_rep1_df.drop(['bin_1','bin_2','bin_3','bin_4'], axis=1)
norm_counts_rep2_df = raw_counts_rep2_df.drop(['bin_1','bin_2','bin_3','bin_4'], axis=1)
norm_counts_rep3_df = raw_counts_rep3_df.drop(['bin_1','bin_2','bin_3','bin_4'], axis=1)

# print(norm_counts_rep1_df)

# create dataframes which only contain read counts and will be used to calculate normalized scores
# raw_counts_only_rep1_df = raw_counts_rep1_df.copy().iloc[:,-4:]
# raw_counts_only_rep2_df = raw_counts_rep2_df.copy().iloc[:,-4:]

raw_counts_only_rep1_df = raw_counts_rep1_df[['bin_1','bin_2','bin_3','bin_4']].copy()
raw_counts_only_rep2_df = raw_counts_rep2_df[['bin_1','bin_2','bin_3','bin_4']].copy()
raw_counts_only_rep3_df = raw_counts_rep3_df[['bin_1','bin_2','bin_3','bin_4']].copy()


# print(norm_counts_rep1_df)
# print(raw_counts_only_rep1_df)

In [21]:
# compute total number of reads per bin (per concentration)
column_sums_rep1 = raw_counts_only_rep1_df.sum().tolist()
print(column_sums_rep1)
# print(list(raw_counts_only_rep1_df.columns))

for bn, bin_no in enumerate(list(raw_counts_only_rep1_df.columns)):
    # normalize by number of reads and fraction of cells in each bin
    norm_counts_rep1_df[bin_no] = raw_counts_only_rep1_df[bin_no] * rep1_cell_fracs[bn]/ column_sums_rep1[bn]
# normalize each row by the sum of each row
norm_counts_rep1_df[raw_counts_only_rep1_df.columns] = (
norm_counts_rep1_df[raw_counts_only_rep1_df.columns]
.div(norm_counts_rep1_df[raw_counts_only_rep1_df.columns].sum(axis=1), axis=0)
)

print(norm_counts_rep1_df[:5])

[285682, 277799, 200775, 228556]
                                         aa_sequence  \
0  HVQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1  HVQLVESGGRLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2  QAQLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3  QVKLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4  QVQLFESGGELVQPGGSLRLSCAASEGNVSMLSLGWFRQAPGQGLE...   

                                        dna_sequence  \
0  [CATGTCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGA...   
1  [CATGTCCAACTGGTCGAGAGCGGTGGGAGACTGGTGCAGCCAGGC...   
2  [CAGGCCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
3  [CAGGTCAAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGC...   
4  [CAGGTCCAACTGTTCGAGAGCGGTGGGGAACTGGTGCAGCCAGGC...   

                                       dna_mutations  \
0  [['G3T', 'A30G', 'G34A', 'G36A', 'G39A', 'C45A...   
1  [['G3T', 'G28A', 'G47A', 'C149T', 'C159T', 'C1...   
2                                          [['T5C']]   
3  [['C7A', 'A30G', 'G34A', 'G36A', 'G39A', 'C159...   
4  [['G13T', 

In [19]:
norm_counts_rep1_df['normalized_score'] = (norm_counts_rep1_df['bin_1']*0 \
                                                                       + norm_counts_rep1_df['bin_2']*(1/3) \
                                                                       + norm_counts_rep1_df['bin_3']*(2/3) \
                                                                       + norm_counts_rep1_df['bin_4']*(3/3))

norm_counts_rep1_df['replicate'] = 'normalized_score_rep1'
print(norm_counts_rep1_df)

                                             aa_sequence  \
0      HVQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1      HVQLVESGGRLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2      QAQLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3      QVKLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4      QVQLFESGGELVQPGGSLRLSCAASEGNVSMLSLGWFRQAPGQGLE...   
...                                                  ...   
33220  QVQLVGSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQVPGQGLE...   
33221  QVQLVGSGGRLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33222  QVQLVKSVGGLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33223  QVQLV_SGGGLIQPGGSLRLSCAASGGNVSMLSLGWYRQAPGQGLE...   
33224  QVQMVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                            dna_sequence  \
0      [CATGTCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGA...   
1      [CATGTCCAACTGGTCGAGAGCGGTGGGAGACTGGTGCAGCCAGGC...   
2      [CAGGCCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
3      [CAGGTCAAACTGGTCGAGAGCGGTGGGGGGC

In [20]:
# compute total number of reads per bin (per concentration)
column_sums_rep2 = raw_counts_only_rep2_df.sum().tolist()
# print(column_sums_rep2)
# print(list(raw_counts_only_rep2_df.columns))

for bn, bin_no in enumerate(list(raw_counts_only_rep2_df.columns)):
    # normalize by number of reads and fraction of cells in each bin
    norm_counts_rep2_df[bin_no] = raw_counts_only_rep2_df[bin_no] * rep2_cell_fracs[bn]/ column_sums_rep2[bn]
# normalize each row by the sum of each row
norm_counts_rep2_df[raw_counts_only_rep2_df.columns] = (
norm_counts_rep2_df[raw_counts_only_rep2_df.columns]
.div(norm_counts_rep2_df[raw_counts_only_rep2_df.columns].sum(axis=1), axis=0)
)

# print(norm_counts_rep2_df)

In [21]:
norm_counts_rep2_df['normalized_score'] = (norm_counts_rep2_df['bin_1']*0 \
                                                                       + norm_counts_rep2_df['bin_2']*(1/3) \
                                                                       + norm_counts_rep2_df['bin_3']*(2/3) \
                                                                       + norm_counts_rep2_df['bin_4']*(3/3))

norm_counts_rep2_df['replicate'] = 'normalized_score_rep2'
# print(norm_counts_rep2_df)

In [22]:
# compute total number of reads per bin (per concentration)
column_sums_rep3 = raw_counts_only_rep3_df.sum().tolist()
# print(column_sums_rep2)
# print(list(raw_counts_only_rep2_df.columns))

for bn, bin_no in enumerate(list(raw_counts_only_rep3_df.columns)):
    # normalize by number of reads and fraction of cells in each bin
    norm_counts_rep3_df[bin_no] = raw_counts_only_rep3_df[bin_no] * rep3_cell_fracs[bn]/ column_sums_rep3[bn]
# normalize each row by the sum of each row
norm_counts_rep3_df[raw_counts_only_rep2_df.columns] = (
norm_counts_rep3_df[raw_counts_only_rep3_df.columns]
.div(norm_counts_rep3_df[raw_counts_only_rep3_df.columns].sum(axis=1), axis=0)
)

norm_counts_rep3_df['normalized_score'] = (norm_counts_rep3_df['bin_1']*0 \
                                                                       + norm_counts_rep3_df['bin_2']*(1/3) \
                                                                       + norm_counts_rep3_df['bin_3']*(2/3) \
                                                                       + norm_counts_rep3_df['bin_4']*(3/3))

norm_counts_rep3_df['replicate'] = 'normalized_score_rep3'

In [23]:
# merge both replicates for amino acid sequences that appear in both replicates
# approach using pivot

norm_counts_long = pd.concat([norm_counts_rep1_df, norm_counts_rep2_df, norm_counts_rep3_df])
# print(norm_counts_long[:2])

norm_counts_binagg = norm_counts_long.groupby('aa_sequence')[['bin_1','bin_2','bin_3','bin_4','sum_reads']].agg(list).reset_index()

# print(norm_counts_binagg)
dupes = norm_counts_long.duplicated(subset=['aa_sequence', 'replicate'], keep=False)
# print(norm_counts_long[dupes].sort_values(['aa_sequence', 'replicate']))
# print(norm_counts_long.shape)

norm_counts_wide = norm_counts_long.pivot(index='aa_sequence', columns='replicate', values='normalized_score').reset_index()
norm_counts_wide.columns.name = '.'
norm_counts_reps = norm_counts_wide.dropna(subset=['normalized_score_rep1','normalized_score_rep2', 'normalized_score_rep3']).reset_index(drop=True)

descriptors = norm_counts_long[['aa_sequence'
                             ,'dna_sequence'
                             , 'dna_mutations'
                             , 'aa_mutations'
                             , 'number_dna_mutations'
                             , 'number_aa_mutations']].drop_duplicates('aa_sequence')

aux_bindata = norm_counts_binagg[['aa_sequence'
                             ,'bin_1'
                             , 'bin_2'
                             , 'bin_3'
                             , 'bin_4'
                             , 'sum_reads']].drop_duplicates('aa_sequence')

norm_counts_description = pd.merge(norm_counts_reps,descriptors, on='aa_sequence', how='left')
norm_counts_all = pd.merge(norm_counts_description,aux_bindata, on='aa_sequence', how='left')

cols = [col for col in norm_counts_all.columns if col not in ['normalized_score_rep1','normalized_score_rep2','normalized_score_rep3']] + ['normalized_score_rep1','normalized_score_rep2','normalized_score_rep3']

norm_counts_final = norm_counts_all[cols]

# print(norm_counts_final)

In [24]:
# remove highly variant sequences differing by 1/3 a.u. -- note that
# differing by 1/3 a.u. means that, on average, the bins
# were sorted into different bins
diff_seqs = []
for idx, row in tqdm.tqdm(norm_counts_final.iterrows(), total = norm_counts_final.shape[0]):
    replicate_difference = row[['normalized_score_rep1', 'normalized_score_rep2', 'normalized_score_rep3']].std()
    if replicate_difference > (math.ceil(1/3*100)/100):
        # print(replicate_difference)
        diff_seqs.append(idx)
norm_counts_uniform = norm_counts_final.copy()
# print(norm_counts_final.loc[diff_seqs])
print(f'removed {len(diff_seqs)} sequences with dissimilar replicate scores')
for idx2 in diff_seqs:    
    norm_counts_uniform = norm_counts_uniform.drop(idx2)

# print(diff_seqs)
# print(len(norm_counts_uniform))

norm_counts_annotated_df = norm_counts_uniform.copy()

# modification to keep dissimilar scores (comment out if not needed)
# norm_counts_annotated_df = norm_counts_final.copy()

# Prepare feature matrix (normalized scores)
X = norm_counts_annotated_df[['normalized_score_rep1', 'normalized_score_rep2','normalized_score_rep3']].values

# Use KMeans clustering to group points into 3 categories
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
kmeans_labels = kmeans.fit_predict(X)

# Map cluster labels to descriptive categories (assign based on mean locations)
cluster_centers = kmeans.cluster_centers_
sorted_indices = np.argsort(cluster_centers[:, 0])  # Sort clusters by normalized_score_rep1
cluster_map = {sorted_indices[0]: 'Inactive',
               sorted_indices[1]: 'Active',
               sorted_indices[2]: 'WT-like'}

# Assign labels to dataframe
norm_counts_annotated_df['predicted_category'] = [cluster_map[label] for label in kmeans_labels]

print(norm_counts_annotated_df)

100%|█████████████████████████████████████████████████████████████████████████████| 3541/3541 [00:01<00:00, 2176.23it/s]


removed 45 sequences with dissimilar replicate scores
                                            aa_sequence  \
0     HVQLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1     QVHLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2     QVQLIESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3     QVQLVASGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4     QVQLVDSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
...                                                 ...   
3536  QVQLVGSGGGLVQPGGSLRLSCAASGGNASMLSLGWFRQAPGQGLE...   
3537  QVQLVGSGGGLVQPGGSLRLSCAASGGNISMLSLGWFRQAPGQGLE...   
3538  QVQLVGSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3539  QVQLVKSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3540  QVRLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                           dna_sequence  \
0     [CATGTCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
1     [CAGGTCCATCTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
2     [CAGGTCCAACTGATCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
3

In [25]:
norm_counts_annotated_df = norm_counts_annotated_df.reset_index(drop=True)
master_rep1_df_unique = master_rep1_df_unique.reset_index(drop=True)
# print(norm_counts_annotated_df.shape)
# print(norm_counts_annotated_df.index)

mask1 = ~master_rep1_df_unique['aa_sequence'].isin(norm_counts_annotated_df['aa_sequence'])
# print(mask.shape)
# print(mask[mask==True])

# print(norm_counts_annotated_df)
removed_seqs_rep1 = master_rep1_df_unique[mask1]
print(removed_seqs_rep1)
print(len(master_rep1_df_unique) -len(norm_counts_annotated_df))

                                             aa_sequence  \
0      HVQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1      HVQLVESGGRLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2      QAQLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3      QVKLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4      QVQLFESGGELVQPGGSLRLSCAASEGNVSMLSLGWFRQAPGQGLE...   
...                                                  ...   
33220  QVQLVGSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQVPGQGLE...   
33221  QVQLVGSGGRLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33222  QVQLVKSVGGLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33223  QVQLV_SGGGLIQPGGSLRLSCAASGGNVSMLSLGWYRQAPGQGLE...   
33224  QVQMVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                            dna_sequence  read_count  \
0      [CATGTCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGA...           3   
1      [CATGTCCAACTGGTCGAGAGCGGTGGGAGACTGGTGCAGCCAGGC...           3   
2      [CAGGCCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...       

In [27]:
norm_counts_annotated_df = norm_counts_annotated_df.reset_index(drop=True)
raw_counts_rep1_df = raw_counts_rep1_df.reset_index(drop=True)
# print(norm_counts_annotated_df.shape)
# print(norm_counts_annotated_df.index)

mask1 = ~raw_counts_rep1_df['aa_sequence'].isin(norm_counts_annotated_df['aa_sequence'])
# print(mask.shape)
# print(mask[mask==True])

# print(norm_counts_annotated_df)
removed_seqs_rep1 = raw_counts_rep1_df[mask1]
print(removed_seqs_rep1)
print(len(raw_counts_rep1_df) -len(norm_counts_annotated_df))

                                             aa_sequence  \
0      HVQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1      HVQLVESGGRLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2      QAQLVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3      QVKLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4      QVQLFESGGELVQPGGSLRLSCAASEGNVSMLSLGWFRQAPGQGLE...   
...                                                  ...   
33220  QVQLVGSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQVPGQGLE...   
33221  QVQLVGSGGRLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33222  QVQLVKSVGGLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33223  QVQLV_SGGGLIQPGGSLRLSCAASGGNVSMLSLGWYRQAPGQGLE...   
33224  QVQMVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                            dna_sequence  \
0      [CATGTCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGA...   
1      [CATGTCCAACTGGTCGAGAGCGGTGGGAGACTGGTGCAGCCAGGC...   
2      [CAGGCCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAGCCAGGC...   
3      [CAGGTCAAACTGGTCGAGAGCGGTGGGGGGC

In [26]:
norm_counts_annotated_df = norm_counts_annotated_df.reset_index(drop=True)
master_rep2_df_unique = master_rep2_df_unique.reset_index(drop=True)
# print(norm_counts_annotated_df.shape)
# print(norm_counts_annotated_df.index)

mask2 = ~master_rep2_df_unique['aa_sequence'].isin(norm_counts_annotated_df['aa_sequence'])
# print(mask.shape)
# print(mask[mask==True])

# print(norm_counts_annotated_df)
removed_seqs_rep2 = master_rep2_df_unique[mask2]
print(removed_seqs_rep2)
print(len(master_rep2_df_unique) -len(norm_counts_annotated_df))

                                              aa_sequence  \
0       HVQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQVPGQGLE...   
1       HVQLVESGGGLVQPGDSLRLSCTASGGNVSMLSLGWFRQAPGQGLE...   
2       HVQLVESGGGLVQPGGSLRLSCAASEGNVSMLSLGWFRQARGQGLE...   
3       HVQLVESGGRLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4       HVRLVERGGGLVQPGGSLRLSCAASGENVSMLSLGWFRQAPGQGLE...   
...                                                   ...   
110086  QV_LVESGGGLIQPGVSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
110087  QV_LVESGGGLVQPGDSLRLSCTASGGNVSMLSLGWLRQAPGQGLE...   
110088  QV_LVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
110089  QV_LVESGGRLIQPGGSLRLSCATTGGNVSMLSLGWFRQAPGQGLE...   
110090  QV_LVESGRGLVQPGDSLRLSCAASGGNISMLSLGWFRQAPGQGLE...   

                                             dna_sequence  read_count  \
0       [CATGTCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGC...           1   
1       [CACGTCCAACTGGTCGAGAGCGGTGGGGGACTGGTACAGCCAGGC...           1   
2       [CACGTCCAACTGGTCGAGAGCGGTGGGGGACTGGTGCAG

In [28]:
norm_counts_annotated_df = norm_counts_annotated_df.reset_index(drop=True)
raw_counts_rep2_df = raw_counts_rep2_df.reset_index(drop=True)
# print(norm_counts_annotated_df.shape)
# print(norm_counts_annotated_df.index)

mask2 = ~raw_counts_rep2_df['aa_sequence'].isin(norm_counts_annotated_df['aa_sequence'])
# print(mask.shape)
# print(mask[mask==True])

# print(norm_counts_annotated_df)
removed_seqs_rep2 = raw_counts_rep2_df[mask2]
print(removed_seqs_rep2)
print(len(raw_counts_rep2_df) -len(norm_counts_annotated_df))

                                             aa_sequence  \
0      QAQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
1      QIQLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
2      QVHLVESGGGLIQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
3      QVHLVESGGGLVQPGDSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
4      QVHLVESGGGLVQPGGSLRLNCTASEGNVSMLSLGWFRQAPGQGLE...   
...                                                  ...   
33126  QVQLVVSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33127  QVQLVVSGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33128  QVQLVVSGGGLVQPGGSLRLSCAVSGGNVSMLSLGWFRQAPGQGLE...   
33129  QVQMVESGGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   
33130  QVRLVESDGGLVQPGGSLRLSCAASGGNVSMLSLGWFRQAPGQGLE...   

                                            dna_sequence  \
0      [CAGGCCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGC...   
1      [CAGATCCAACTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGC...   
2      [CAGGTCCACCTGGTCGAGAGCGGTGGGGGGCTGATACAACCAGGC...   
3      [CAGGTCCACCTGGTCGAGAGCGGTGGGGGAC